# Foto-Migration: Google Drive + OneDrive → iCloud Photos (Hybrid)

**Ablauf:** Colab konvertiert (JPEG/MP4, EXIF erhalten) → fertige Dateien in Google-Drive-Ordner `Fertig_iCloud` → iPhone-Kurzbefehl importiert in Fotos-App → iCloud Photos.

**Zellen der Reihe nach ausführen (▶).** Details: `SETUP.md` im Repo.


## 1) Google Drive einbinden
Fenster bestätigen. Danach liegen deine Drive-Dateien unter `/content/drive/MyDrive`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2) Systemtools + Python-Pakete installieren
exiftool (Metadaten) und ffmpeg (Video) via apt, Rest via pip.


In [ ]:
!apt-get -qq update
!apt-get -qq install -y exiftool ffmpeg >/dev/null
!exiftool -ver && ffmpeg -version | head -n1


## 3) Code holen (privates Repo)
GitHub-Token aus **Colab-Secrets** (Schlüssel-Symbol links) mit Namen `GH_TOKEN` (fein-granular, nur Lese-Recht auf dieses Repo).
`REPO_URL` unten auf dein Repo anpassen (ohne `https://`).


In [ ]:
from google.colab import userdata
GH_TOKEN = userdata.get('GH_TOKEN')
REPO_URL = 'github.com/pasquales-art/foto-migration-icloud.git'  # <-- anpassen
REPO_DIR = '/content/foto-migration-icloud'
import os, shutil
if os.path.exists(REPO_DIR): shutil.rmtree(REPO_DIR)
!git clone https://{GH_TOKEN}@{REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip -q install -r requirements.txt


## 4) Secrets & Einstellungen setzen
OneDrive-Werte aus Colab-Secrets (`MS_CLIENT_ID`). Einstellungen hier direkt setzen.
**Wichtig:** Erst `DELETE_STAGE='dry_run'` lassen und Log prüfen, bevor du löschst!


In [ ]:
import os
# --- OneDrive (nur falls genutzt) ---
os.environ['MS_CLIENT_ID'] = userdata.get('MS_CLIENT_ID') or ''
os.environ['MS_TENANT']    = 'consumers'

# --- Quellen an/aus ---
os.environ['USE_GDRIVE']   = 'true'
os.environ['USE_ONEDRIVE'] = 'true'   # auf 'false', wenn (noch) kein OneDrive-Setup

# --- Pfade ---
os.environ['GDRIVE_MOUNT']  = '/content/drive/MyDrive'
os.environ['GDRIVE_SUBDIR'] = 'Bilder'
os.environ['ONEDRIVE_PATH'] = 'Meine Daten/Bilder'
os.environ['READY_SUBDIR']  = 'Fertig_iCloud'
os.environ['BASE_DIR']      = '/content/drive/MyDrive/Fertig_iCloud/_arbeit'  # State/Logs überleben Neustart

# --- Verhalten ---
os.environ['TEST_MODE']    = 'true'      # ERST testen!
os.environ['TEST_LIMIT']   = '100'
os.environ['FALLBACK_DATE']= '2022-12-01'
os.environ['DELETE_STAGE'] = 'dry_run'   # dry_run -> trash -> permanent


## 5) Lauf starten
Bei OneDrive: einmal Login-URL + Code bestätigen (erscheint in der Ausgabe).


In [ ]:
import importlib, run
importlib.reload(run)
run.main()


## 6) Ergebnis prüfen
- Fertige Dateien: `MyDrive/Fertig_iCloud`
- Log: `.../_arbeit/logs/migration.log`

Wenn Testlauf gut → `TEST_MODE='false'` und erneut Zelle 5. Löschen erst `trash`, dann `permanent`.


In [ ]:
!tail -n 30 /content/drive/MyDrive/Fertig_iCloud/_arbeit/logs/migration.log


## 7) iPhone: Import in iCloud Photos
Öffne den Kurzbefehl aus `SHORTCUT.md` am iPhone → importiert `Fertig_iCloud` in die Fotos-App.
Dank gesetztem EXIF-Datum landet alles korrekt sortiert in iCloud Photos.
